# **Install Required Libraries**

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl scikit-learn
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


# **Import Required Libraries**

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig, AutoModelForSequenceClassification
import torch
import evaluate

# **Load TweetEval Dataset**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("tweet_eval", "sentiment")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


# **Load Pretrained Model & Tokenizer**

In [ ]:
model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

# **Tokenization Function**

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

encoded_dataset = dataset.map(tokenize_function, batched=True)

encoded_dataset = encoded_dataset.remove_columns(["text"])
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format("torch")

Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

# **Load Baseline Model**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

model.config.pad_token_id = tokenizer.eos_token_id

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Phi3ForSequenceClassification LOAD REPORT from: microsoft/Phi-3-mini-4k-instruct
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# **Define Evaluation Metrics**

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_score = f1.compute(predictions=predictions, references=labels, average="macro")

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

# **Baseline Evaluation**

In [ ]:
small_test = encoded_dataset["test"].select(range(2000))

trainer = Trainer(
    model=model,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)

baseline_results = trainer.evaluate()
print("Baseline Results:", baseline_results)

Baseline Results: {'eval_loss': 1.2882031202316284, 'eval_model_preparation_time': 0.0036, 'eval_accuracy': 0.3385, 'eval_f1': 0.2635636932525734, 'eval_runtime': 969.1003, 'eval_samples_per_second': 2.064, 'eval_steps_per_second': 0.258}


# **QLoRA Preparation**

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,      # Better memory efficiency
    bnb_4bit_quant_type="nf4",           # Recommended for QLoRA
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True              # VERY IMPORTANT
)

model.config.pad_token_id = tokenizer.eos_token_id

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Phi3ForSequenceClassification LOAD REPORT from: microsoft/Phi-3-mini-4k-instruct
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# **Parameter Efficient Fine-Tuning**

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["qkv_proj", "o_proj"],
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 4,727,808 || all params: 3,727,315,968 || trainable%: 0.1268


# **Define Training Arguments**

In [ ]:
small_train = encoded_dataset["train"].shuffle(seed=42).select(range(10000))
small_val = encoded_dataset["validation"].shuffle(seed=42).select(range(2000))
small_test = encoded_dataset["test"].shuffle(seed=42).select(range(3000))

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,     # reduced
    weight_decay=0.01,
    report_to=[],
    fp16=False
)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer)

# **Trainer Setup**

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# **Train Model (QLoRA Fine-Tuning)**

In [ ]:
trainer.train()

Step,Training Loss
500,4.302039


TrainOutput(global_step=625, training_loss=4.235025, metrics={'train_runtime': 2854.6636, 'train_samples_per_second': 3.503, 'train_steps_per_second': 0.219, 'total_flos': 2.78693019648e+16, 'train_loss': 4.235025, 'epoch': 1.0})

# **Final Evaluation**

In [ ]:
results = trainer.evaluate(small_test)
print("Final Test Results:", results)

Final Test Results: {'eval_loss': 1.197335958480835, 'eval_accuracy': 0.4066666666666667, 'eval_f1': 0.3023207780422501, 'eval_runtime': 425.4688, 'eval_samples_per_second': 7.051, 'eval_steps_per_second': 1.763, 'epoch': 1.0}
